# Hotel Operations Performance Analysis

Use this notebook after generating data and loading PostgreSQL tables. The analysis should validate Power BI KPIs, identify seasonal patterns, and produce final numbers for the README Key Findings section.

In [ ]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path('../data/raw')
bookings = pd.read_csv(DATA_DIR / 'bookings.csv', parse_dates=['check_in_date', 'check_out_date'])
daily_sales = pd.read_csv(DATA_DIR / 'daily_sales.csv', parse_dates=['sales_date'])
expenses = pd.read_csv(DATA_DIR / 'department_expenses.csv', parse_dates=['expense_month'])
ar = pd.read_csv(DATA_DIR / 'accounts_receivable.csv')


In [ ]:
monthly_revenue = (
    daily_sales.assign(month=daily_sales['sales_date'].dt.to_period('M').dt.to_timestamp())
    .groupby('month', as_index=False)['amount_jpy']
    .sum()
    .rename(columns={'amount_jpy': 'revenue_jpy'})
)
monthly_cost = (
    expenses.groupby('expense_month', as_index=False)['amount_jpy']
    .sum()
    .rename(columns={'expense_month': 'month', 'amount_jpy': 'operating_cost_jpy'})
)
profit = monthly_revenue.merge(monthly_cost, on='month', how='left')
profit['profit_margin_pct'] = (profit['revenue_jpy'] - profit['operating_cost_jpy']) / profit['revenue_jpy'] * 100
profit.sort_values('profit_margin_pct', ascending=False).head()

In [ ]:
ar['aging_bucket'] = pd.cut(
    ar['days_past_due'],
    bins=[-1, 30, 60, float('inf')],
    labels=['0-30 days', '31-60 days', '60+ days'],
)
ar.loc[ar['outstanding_balance_jpy'].eq(0), 'aging_bucket'] = 'Paid'
ar.groupby('aging_bucket', observed=False)['outstanding_balance_jpy'].agg(['count', 'sum'])